## PyG 核心依赖库

### 1. torch_geometric.utils
**用途**：图数据预处理工具函数
### 2. torch_scatter
**用途**：图消息传递中的邻居特征聚合（底层 C++/CUDA 优化）
### 3. torch_sparse
**用途**：高效存储和运算大规模稀疏邻接矩阵，支持 GPU 加速

In [1]:
import torch
from torch_sparse import SparseTensor
from torch_geometric.utils import degree
from torch_scatter import scatter_mean, scatter_max, scatter_min

# --- 模拟数据 ---
# 假设我们有 4 个节点 (编号 0, 1, 2, 3)
print("边的连接情况: 0->1, 1->2, 2->0, 2->3 (这是一个有向图)")
edge_index = torch.tensor([[0, 1, 2, 2],  # 起始节点 (行)
                           [1, 2, 0, 3]]) # 终止节点 (列)
num_nodes = 4
src, dst = edge_index

out_degree = degree(src, num_nodes=num_nodes, dtype=torch.float)
in_degree = degree(dst, num_nodes=num_nodes, dtype=torch.float)
total_degree = in_degree + out_degree

print(f"各节点度数：（从0号节点 -> 3号节点）")
print(f"出度：{out_degree}， 入度：{in_degree}， 总度：{total_degree}")

# 构建邻接矩阵
adj = SparseTensor(row=src, col=dst, sparse_sizes=(num_nodes, num_nodes))
# 计算邻居的 total_degree, 下一步计算需要二维
total_degree_2d = total_degree.unsqueeze(-1)

neigh_degree_sum = adj.matmul(total_degree_2d).squeeze(-1)
print(f"\n计算每个节点周围邻居的度之和:\n{neigh_degree_sum}")

neigh_degree_mean = neigh_degree_sum / (total_degree + 1e-6)
print(f"\n计算每个节点周围邻居的度数平均值:\n{neigh_degree_mean}")

src_deg = total_degree[src] # 获取每条边起始节点的度数
max_neigh_degree, _ = scatter_max(src_deg, dst, dim=0, dim_size=num_nodes) # 求指向dst的节点的度数最大值
print(f"\n计算每个节点周围邻居的度数最大值:\n{max_neigh_degree}")


e:\Python\py\envs\pytorch\lib\site-packages\torch\cuda\__init__.py:51: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


边的连接情况: 0->1, 1->2, 2->0, 2->3 (这是一个有向图)
各节点度数：（从0号节点 -> 3号节点）
出度：tensor([1., 1., 2., 0.])， 入度：tensor([1., 1., 1., 1.])， 总度：tensor([2., 2., 3., 1.])

计算每个节点周围邻居的度之和:
tensor([2., 3., 3., 0.])

计算每个节点周围邻居的度数平均值:
tensor([1.0000, 1.5000, 1.0000, 0.0000])

计算每个节点周围邻居的度数最大值:
tensor([3., 2., 2., 3.])


tensor([3., 2., 2., 3.])

In [42]:
a = torch.tensor([[1, 2, 3], [1, 2, 3]])
a.view(-1, 2, 1)


tensor([[[1],
         [2]],

        [[3],
         [1]],

        [[2],
         [3]]])